# Erken matların teşhisi: kayıtlı pozisyonlar, yeni maç yok

Dört en erken kayıp (**19, 55, 77, 76**) ile iki galibiyet (**20, 56**) ve
iki beraberlik (**96, 37**) seçilir. Kayıplardaki bütün aday hamleleri,
karşılaştırma oyunlarının ilk sekizer aday hamlesi incelenir. Geçmiş hamleler
korunur; referans, TT ve paketli değerlendirmenin açık olduğu sabit main'dir.

Stockfish her konumu ve gerçekten oynanan hamleyi iki bütçeyle inceler:
istenen derinlik 12/18, her sorguda en fazla 100.000 düğüm. Ulaşılan gerçek
derinlik kaydedilir. Ardından Acemi normal ve NullMove+LMR kapalı olarak
12.000 düğüm / en fazla 8 derinlikle hamle seçer; bunlar da Stockfish ile etiketlenir.
Her sorgu temiz TT ve Seed=1 kullanır. Bu soğuk pozisyon teşhisi, maçtaki
sıcak TT ve ilerlemiş rastgelelik durumunun bire bir tekrarı değildir.

**Yeni maç oynanmaz.** Bu ölçüm ilk hatanın kesin kanıtı veya Elo testi değildir.
Sonuçlar GitHub'a gider; sonraki kod değişikliğini bu kayıtlardan seçeceğiz.
`codex/tactical-diagnostics` dalından içe alın. Kaggle Internet ve Contents
_yazma_ yetkili GITHUB_TOKEN secret'ı açık olsun. Run All çalıştırın.


In [ ]:
GITHUB_REPO_URL = "https://github.com/xdxddxdxd/acemi.git"
GITHUB_BRANCH = "codex/tactical-diagnostics"
WORKSPACE = "/kaggle/working/acemi-tactical-diagnostics"
SESSION_ID = "tactical-losses-v1"
ENGINE_COMMIT = "38b0722b084552d8e94eb62e3558d447b641969c"
ENGINE_SOURCE = "/kaggle/working/acemi-tactical-engine"
RETRY_INCOMPLETE = False


In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("This diagnostic notebook runs only in Kaggle.")

def checked(args, *, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode:
        raise RuntimeError(f"Command failed: {args[0]}\n{(result.stdout + result.stderr)[-4000:]}")
    return result.stdout.strip()

workspace = Path(WORKSPACE)
if not workspace.exists():
    checked(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(workspace)])
else:
    checked(["git", "-C", str(workspace), "fetch", "origin", GITHUB_BRANCH])
    local = subprocess.run(["git", "-C", str(workspace), "show-ref", "--verify", "--quiet", f"refs/heads/{GITHUB_BRANCH}"])
    if local.returncode == 0:
        checked(["git", "-C", str(workspace), "switch", GITHUB_BRANCH])
    else:
        checked(["git", "-C", str(workspace), "switch", "--create", GITHUB_BRANCH, "--track", f"origin/{GITHUB_BRANCH}"])
    checked(["git", "-C", str(workspace), "merge", "--ff-only", f"origin/{GITHUB_BRANCH}"])
os.chdir(workspace)
if not (workspace / "tools/tactical_diagnostics.py").exists():
    raise RuntimeError("Import the latest notebook from codex/tactical-diagnostics.")
checked([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-tools.txt"], cwd=workspace)
stockfish_bin = shutil.which("stockfish") or ("/usr/games/stockfish" if Path("/usr/games/stockfish").exists() else None)
if stockfish_bin is None:
    checked(["apt-get", "update", "-qq"])
    checked(["apt-get", "install", "-y", "-qq", "stockfish"])
    stockfish_bin = shutil.which("stockfish") or ("/usr/games/stockfish" if Path("/usr/games/stockfish").exists() else None)
if stockfish_bin is None:
    raise RuntimeError("Stockfish executable was not found after installation.")
engine_source = Path(ENGINE_SOURCE)
if not engine_source.exists():
    checked(["git", "clone", "--depth", "1", "--no-checkout", GITHUB_REPO_URL, str(engine_source)])
checked(["git", "-C", str(engine_source), "fetch", "--depth", "1", "origin", ENGINE_COMMIT])
checked(["git", "-C", str(engine_source), "checkout", "--detach", ENGINE_COMMIT])
if checked(["git", "-C", str(engine_source), "rev-parse", "HEAD"]) != ENGINE_COMMIT:
    raise RuntimeError("Pinned engine mismatch")
session = workspace / "data" / "tactics" / SESSION_ID
print("Engine:", ENGINE_COMMIT, "Stockfish:", stockfish_bin)
print("Recorded-position diagnostics only; new matches: 0; results:", session)


In [ ]:
# Publish records only. The token is never written to a file or a remote URL.
import base64
from kaggle_secrets import UserSecretsClient

def publish_results(message):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    if not token:
        raise RuntimeError("Add GITHUB_TOKEN under Kaggle Add-ons / Secrets.")
    auth = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = os.environ.copy()
    env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
                "GIT_CONFIG_VALUE_0": auth, "GIT_TERMINAL_PROMPT": "0"})
    def git(*args, check=True):
        result = subprocess.run(["git", *args], cwd=workspace, env=env, text=True, capture_output=True)
        if check and result.returncode:
            details = (result.stdout + result.stderr).replace(token, "[REDACTED]").replace(auth, "[REDACTED]")
            raise RuntimeError(f"Git {args[0]} failed: {details[-2000:]}")
        return result
    if git("branch", "--show-current").stdout.strip() != GITHUB_BRANCH:
        raise RuntimeError("Wrong results branch; rerun repository setup.")
    git("config", "user.name", "Acemi Kaggle A-B")
    git("config", "user.email", "acemi-kaggle-runner@users.noreply.github.com")
    paths = [str(session.relative_to(workspace))]
    git("add", "--", *paths)
    if git("diff", "--cached", "--quiet", check=False).returncode:
        git("commit", "-m", message)
    git("fetch", "origin", GITHUB_BRANCH)
    if int(git("rev-list", "--count", "HEAD..FETCH_HEAD").stdout.strip()):
        git("rebase", "FETCH_HEAD")
    git("push", "origin", f"HEAD:{GITHUB_BRANCH}")
    print("Published:", message)


In [ ]:
# Fresh processes avoid any old tools/engine modules in the Kaggle kernel.
print(checked([sys.executable, "-m", "unittest", "tests.test_tactical_diagnostics", "-v"], cwd=workspace))
print(checked([sys.executable, "-u", "-m", "tools.tactical_diagnostics", "prepare", "--id", SESSION_ID,
               "--engine-source", str(engine_source), "--stockfish", stockfish_bin], cwd=workspace))
plan = json.loads((session / "protocol.json").read_text(encoding="utf-8"))
print("Frozen positions:", plan["positions"], "selected games:", plan["selected_games"])
publish_results(f"data: prepare recorded tactical diagnosis ({SESSION_ID})")


In [ ]:
command = [sys.executable, "-u", "-m", "tools.tactical_diagnostics", "run", "--session", str(session)]
if RETRY_INCOMPLETE:
    command.append("--retry-incomplete")
result = subprocess.run(command, cwd=workspace)
publish_results(f"data: recorded tactical diagnosis results ({SESSION_ID})")
if result.returncode:
    raise RuntimeError("Diagnosis stopped; completed positions and error log were retained. Inspect logs before retrying.")
print((session / "SUMMARY.md").read_text(encoding="utf-8"))


In [ ]:
# Safe push retry only: this cell never starts an engine or match.
publish_results(f"data: finalize recorded tactical diagnosis ({SESSION_ID})")
if (session / "SUMMARY.md").exists():
    print((session / "SUMMARY.md").read_text(encoding="utf-8"))
print("New matches: 0. The saved Stockfish labels will be analysed separately.")
